# Tutorial 8 — Optimizers: SGD, Adam, AdamW & Batch Size

**Series:** Training Language Models from Scratch: A Hacker's Guide  
**Part III — Pretraining**  
**Follows:** Tutorial 7 (Data Pipelines for Pretraining)  
**Precedes:** Tutorial 9 (Pretraining: Mixed Precision, Schedules & Scaling Laws)

---

## What This Tutorial Covers

Every tutorial in this series calls `optimizer.step()`. Until now, we have
treated it as a black box. This tutorial opens it.

The choice of optimizer, learning rate, and batch size are the three levers
that most determine whether a training run converges, diverges, or plateaus.
They are also the three that practitioners most often get wrong by copying
defaults without understanding them.

This tutorial covers:

1. **SGD and why it fails at scale** — the loss landscape of a transformer is
   high-dimensional and ill-conditioned. Core SGD cannot navigate it.
2. **Momentum** — a physically motivated fix: accumulating velocity in
   parameter space to smooth out noisy gradient directions.
3. **Adam** — per-parameter adaptive learning rates derived from gradient
   history. The full update rule, bias correction, and why it works where
   SGD with momentum does not.
4. **AdamW** — why Adam's built-in weight decay is mathematically wrong and
   how AdamW fixes it. The argument from first principles.
5. **Batch size** — what it controls beyond throughput: the gradient noise
   scale, the critical batch size, and why bigger is not always better.
6. **Choosing a learning rate** — the learning rate range test, the linear
   scaling rule, and the heuristics that actually work for LLM fine-tuning.
7. **Gradient accumulation** — simulating a large batch on a small GPU.
   The correct implementation, the silent bug, and how it interacts with
   mixed precision.
8. **Optimizer hyperparameters for fine-tuning vs pretraining** — different
   regimes, different settings. Why the defaults that work for pretraining
   can destroy a fine-tuning run.

---

## 1. SGD — The Baseline

Stochastic gradient descent is the simplest update rule. At each step, take
a mini-batch of $B$ samples, compute the gradient of the loss with respect to
all parameters, and move in the negative gradient direction:

$$\theta_{t+1} = \theta_t - \eta \cdot g_t$$

where $g_t = \nabla_\theta \mathcal{L}(\theta_t; \mathcal{B}_t)$ is the
gradient on mini-batch $\mathcal{B}_t$ and $\eta$ is the learning rate.

In [ ]:
import torch
import torch.nn as nn

# Manual SGD to make the update rule explicit
model = nn.Linear(128, 64)
lr = 0.01

x = torch.randn(32, 128)
y = torch.randn(32, 64)

pred = model(x)
loss = nn.functional.mse_loss(pred, y)
loss.backward()

with torch.no_grad():
    for p in model.parameters():
        p -= lr * p.grad    # θ ← θ - η·g
        p.grad.zero_()

# Equivalent with torch.optim.SGD
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
optimizer.zero_grad()
pred = model(x)
loss = nn.functional.mse_loss(pred, y)
loss.backward()
optimizer.step()

### Why SGD fails for language models

The loss surface of a transformer is [**ill-conditioned**]{.underline}: gradients are
much larger in some directions than others. Consider a simple 2D example
with loss $\mathcal{L}(\theta_1, \theta_2) = \theta_1^2 + 100\,\theta_2^2$.
The gradient is $(2\theta_1, 200\theta_2)$ — the curvature in the
$\theta_2$ direction is 100× larger. Any learning rate large enough to make
progress in $\theta_1$ will cause oscillation in $\theta_2$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simulate SGD on an ill-conditioned quadratic
theta = np.array([10.0, 1.0])   # starting point
lr    = 0.009                    # largest stable lr for θ₂

path_sgd = [theta.copy()]
for _ in range(100):
    grad  = np.array([2 * theta[0], 200 * theta[1]])
    theta = theta - lr * grad
    path_sgd.append(theta.copy())

path_sgd = np.array(path_sgd)

# Plot: notice the oscillation in the θ₂ direction
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(path_sgd[:, 0], label='θ₁')
axes[0].plot(path_sgd[:, 1], label='θ₂')
axes[0].set(title='SGD: parameter trajectories', xlabel='step')
axes[0].legend()

axes[1].plot(path_sgd[:, 0], path_sgd[:, 1], '-o', markersize=2)
axes[1].set(title='SGD: trajectory in parameter space',
            xlabel='θ₁', ylabel='θ₂')
plt.tight_layout(); plt.show()

The zigzagging is not just aesthetically unpleasant — it wastes gradient
evaluations and slows convergence. In a 100M-parameter transformer, the
condition number of the loss Hessian is orders of magnitude larger than 100.
SGD simply cannot converge in reasonable time.

---

## 2. SGD with Momentum

Momentum replaces the raw gradient with an exponential moving average
(EMA) — a velocity vector $v_t$ that accumulates gradient history:

$$v_t = \mu \cdot v_{t-1} + g_t$$
$$\theta_{t+1} = \theta_t - \eta \cdot v_t$$

The scalar $\mu \in [0, 1)$ is the **momentum coefficient** (typical value:
0.9). When $\mu = 0$, this reduces to plain SGD.

### The physical intuition

Think of the parameter as a ball rolling down a loss surface.
The gradient gives instantaneous force; $v_t$ is velocity.
On steep slopes (large $g_t$), velocity builds up.
In flat regions or when the gradient sign flips (oscillating ravine),
the velocity damps the oscillation because the positive and negative
gradient contributions partially cancel in the EMA.

In [ ]:
# Manual SGD + momentum
mu = 0.9
velocity = {id(p): torch.zeros_like(p) for p in model.parameters()}

for step in range(1000):
    optimizer_zero_grad()
    loss = compute_loss()
    loss.backward()

    with torch.no_grad():
        for p in model.parameters():
            v = velocity[id(p)]
            v.mul_(mu).add_(p.grad)       # v ← μv + g
            p.add_(-lr * v)               # θ ← θ - η·v
            p.grad.zero_()

# torch.optim.SGD with momentum
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

### Why momentum still fails for transformers

Momentum addresses oscillation but not the core problem: different
parameters need different learning rates. The embedding parameters of a
large vocabulary need small updates (they are updated by many samples);
the output projection parameters need larger updates early in training.
A single global learning rate cannot handle this.

The fix requires **per-parameter adaptive learning rates** — which is
exactly what Adam provides.

---

## 3. Adam — Adaptive Moment Estimation

Adam (Kingma & Ba, 2015) maintains two exponential moving averages
per parameter:

- **First moment** $m_t$ — the running mean of the gradient (direction)
- **Second moment** $v_t$ — the running mean of the squared gradient (magnitude)

$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$$
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$$

Both are initialized to zero, so at the first steps they are biased
toward zero. The **bias-corrected** estimates are:

$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

The parameter update is:

$$\theta_{t+1} = \theta_t - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

where $\epsilon \approx 10^{-8}$ prevents division by zero.

### What the update rule actually does

The denominator $\sqrt{\hat{v}_t}$ adapts the effective step size
per-parameter:

- If $g_t$ has been **consistently large** (high $\hat{v}_t$), the
  step size shrinks. The optimizer is cautious in high-curvature directions.
- If $g_t$ has been **small or noisy** (low $\hat{v}_t$), the step size
  grows. The optimizer moves faster in flat, noisy directions.

The numerator $\hat{m}_t$ provides the momentum smoothing effect from SGD
with momentum.

The result: Adam effectively applies a **different local learning rate
to each parameter** based on that parameter's gradient history.

In [ ]:
import torch

# Manual Adam to show every operation
class ManualAdam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        self.params = list(params)
        self.lr     = lr
        self.beta1, self.beta2 = betas
        self.eps    = eps
        self.t      = 0
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]

    def step(self):
        self.t += 1
        for i, p in enumerate(self.params):
            if p.grad is None:
                continue
            g = p.grad

            # Update biased moment estimates
            self.m[i].mul_(self.beta1).add_(g,          alpha=1 - self.beta1)
            self.v[i].mul_(self.beta2).addcmul_(g, g,   value=1 - self.beta2)

            # Bias correction
            m_hat = self.m[i] / (1 - self.beta1 ** self.t)
            v_hat = self.v[i] / (1 - self.beta2 ** self.t)

            # Parameter update
            p.data.addcdiv_(m_hat, v_hat.sqrt().add_(self.eps), value=-self.lr)

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

### Default hyperparameters

| Hyperparameter | Default | What it controls |
|---|---|---|
| `lr` (η) | 1e-3 | Overall step size magnitude |
| `beta1` (β₁) | 0.9 | Momentum of gradient direction |
| `beta2` (β₂) | 0.999 | Momentum of gradient magnitude |
| `eps` (ε) | 1e-8 | Numerical stability |

[[For language model training, `beta2=0.95` (GPT-3/NanoGPT default) is
often preferred over `0.999`.]{.mark} A lower `beta2` means the second moment
forgets old magnitudes faster, giving the optimizer more flexibility to
adapt to changing gradient scales across training phases.

In [ ]:
# Standard Adam for language models
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=3e-4,
    betas=(0.9, 0.95),   # note: 0.95 not 0.999 for LLMs
    eps=1e-8,
)

[[**NOTE:** The "memory length" of an EMA is roughly $1/(1 - \beta_2)$ steps.]{.underline} So `0.999` retains ~1000 steps of gradient magnitude history, while `0.95` retains only ~20. For long pretraining runs where gradient magnitudes shift significantly across phases (early chaos → stable convergence → late refinement), the shorter memory makes the adaptive scaling more responsive to recent behavior.

---

## 4. AdamW — Fixing Weight Decay

### The problem with Adam + L2 regularization

A natural desire during training is **weight decay**: penalizing large
weights to improve generalization. The standard way is L2 regularization —
adding $\lambda \|\theta\|^2$ to the loss, which adds $2\lambda\theta$ to
the gradient.

For SGD, L2 regularization and weight decay are mathematically equivalent:
both produce an update of the form $\theta_{t+1} = (1 - 2\eta\lambda)\theta_t - \eta g_t$.

[For Adam, they are **not equivalent**.]{.mark} When you add $\lambda\theta$ to the
gradient, it gets processed through the adaptive scaling mechanism:

$$\theta_{t+1} = \theta_t - \eta \cdot \frac{\hat{m}_t + \lambda\hat{\theta}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

The weight decay term is **divided by the adaptive denominator** $\sqrt{\hat{v}_t}$.
This means that parameters with large gradient history (large $\hat{v}_t$, which
tend to be the important parameters that get large gradient updates) have their
weight decay **suppressed**. The regularization is weaker exactly where you
want it strongest.

In practice, vanilla Adam + L2 regularization leads to models that overfit
more than expected — the effective regularization is inconsistent across
parameters.

### The AdamW fix

AdamW (Loshchilov & Hutter, 2019) decouples weight decay from the gradient
update. Instead of adding the weight decay to the gradient *before* adaptive
scaling, it applies it *directly* to the parameters *after* the adaptive update:

$$\theta_{t+1} = \theta_t - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} - \eta \lambda \theta_t$$

[The weight decay term $\eta\lambda\theta_t$ is pure **shrinkage**]{.mark} — it
scales $\theta$ toward zero by a constant factor at every step, regardless
of gradient history. Every parameter experiences the same regularization
pressure, proportional only to its current magnitude.

In [ ]:
# AdamW pseudocode — note where weight_decay is applied
class ManualAdamW:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999),
                 eps=1e-8, weight_decay=0.01):
        self.params       = list(params)
        self.lr           = lr
        self.beta1, self.beta2 = betas
        self.eps          = eps
        self.weight_decay = weight_decay
        self.t = 0
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]

    def step(self):
        self.t += 1
        for i, p in enumerate(self.params):
            if p.grad is None:
                continue
            g = p.grad

            # Moment estimates (gradient only — no weight decay here)
            self.m[i].mul_(self.beta1).add_(g,        alpha=1 - self.beta1)
            self.v[i].mul_(self.beta2).addcmul_(g, g, value=1 - self.beta2)

            m_hat = self.m[i] / (1 - self.beta1 ** self.t)
            v_hat = self.v[i] / (1 - self.beta2 ** self.t)

            # Adaptive update (same as Adam)
            p.data.addcdiv_(m_hat, v_hat.sqrt().add_(self.eps), value=-self.lr)

            # Weight decay — applied SEPARATELY, not through adaptive scaling
            p.data.mul_(1 - self.lr * self.weight_decay)

In [ ]:
# PyTorch AdamW
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    betas=(0.9, 0.95),
    eps=1e-8,
    weight_decay=0.1,    # standard for LLM pretraining
)

### Comparing Adam, Adam+L2, and AdamW

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

def train_and_eval(optimizer_fn, steps=500, seed=42):
    torch.manual_seed(seed)
    model = nn.Sequential(nn.Linear(64, 128), nn.ReLU(),
                          nn.Linear(128, 64), nn.ReLU(),
                          nn.Linear(64, 10))
    opt = optimizer_fn(model.parameters())

    torch.manual_seed(seed)
    X = torch.randn(1000, 64); Y = torch.randint(0, 10, (1000,))
    X_val = torch.randn(200, 64); Y_val = torch.randint(0, 10, (200,))

    train_losses, val_losses, weight_norms = [], [], []
    for step in range(steps):
        idx = torch.randint(0, 1000, (64,))
        opt.zero_grad()
        logits = model(X[idx])
        loss   = nn.functional.cross_entropy(logits, Y[idx])
        loss.backward()
        opt.step()

        train_losses.append(loss.item())
        with torch.no_grad():
            val_loss = nn.functional.cross_entropy(model(X_val), Y_val)
            val_losses.append(val_loss.item())
            wn = sum(p.norm().item()**2 for p in model.parameters())**0.5
            weight_norms.append(wn)

    return train_losses, val_losses, weight_norms

results = {
    'Adam':       train_and_eval(lambda p: torch.optim.Adam(p, lr=3e-4)),
    'Adam+L2':    train_and_eval(lambda p: torch.optim.Adam(
                                     p, lr=3e-4, weight_decay=0.01)),
    'AdamW':      train_and_eval(lambda p: torch.optim.AdamW(
                                     p, lr=3e-4, weight_decay=0.01)),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for name, (tl, vl, wn) in results.items():
    axes[0].plot(tl, label=name, alpha=0.7)
    axes[1].plot(vl, label=name, alpha=0.7)
    axes[2].plot(wn, label=name, alpha=0.7)

for ax, title in zip(axes, ['Train Loss', 'Val Loss', 'Weight Norm']):
    ax.set(title=title, xlabel='step'); ax.legend()
plt.tight_layout(); plt.show()

# Expected: AdamW maintains lower, more consistent weight norms than Adam+L2
# and generalizes better (lower val loss) than vanilla Adam

### Which parameters to apply weight decay to

[Not all parameters should be weight-decayed:]{.underline}

- **Decayed:** weight matrices (`nn.Linear` weights, attention projections,
  FFN weights). These are the parameters that benefit from regularization.
- **Not decayed:** bias terms, layer norm scale/shift parameters, embeddings.
  These are low-dimensional or serve as offsets — regularizing them hurts.

In [ ]:
def make_optimizer(model, lr, weight_decay, betas=(0.9, 0.95)):
    """Separate parameter groups: decay weights, don't decay biases/norms."""
    decay_params, no_decay_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim >= 2:          # weight matrices
            decay_params.append(p)
        else:                    # biases, layer norm params (1-D tensors)
            no_decay_params.append(p)

    param_groups = [
        {'params': decay_params,    'weight_decay': weight_decay},
        {'params': no_decay_params, 'weight_decay': 0.0},
    ]
    return torch.optim.AdamW(param_groups, lr=lr, betas=betas)

This pattern is used by every serious LLM training codebase (GPT-NeoX,
NanoGPT, LLaMA, Mistral).

---

## 5. Batch Size — What It Actually Controls

Batch size is typically chosen based on what fits in GPU memory, then
forgotten. It deserves more thought, because it controls the quality of
the gradient estimate, which determines stable learning rate range,
convergence speed per epoch, and the tradeoff between compute efficiency
and generalization.

### The gradient noise scale

When you estimate the gradient on a mini-batch of $B$ samples, you are
estimating the true gradient (the full-dataset gradient) with noise.
The **gradient noise scale** $B^*$ is the batch size at which the gradient
estimate becomes "reliable enough":

<!--@c1773852616106-->
$$B^* = \frac{\text{tr}(\Sigma)}{\|g_{\text{true}}\|^2}$$

where $\Sigma$ is the covariance of the per-sample gradients and
$g_{\text{true}}$ is the true gradient.

- When $B \ll B^*$: gradients are noisy. Small batches explore the loss
  landscape stochastically — helpful early in training but inefficient late.
- When $B \gg B^*$: you are computing gradient estimates more accurately
  than necessary. Extra samples give diminishing returns.
- When $B \approx B^*$: [the **critical batch size**]{.mark} — maximum gradient
  quality per FLOP.

For most LLM training, $B^*$ is in the range of 100K–500K tokens per batch.
Smaller models have smaller $B^*$; larger models have larger $B^*$.

### The linear scaling rule

When you double the batch size, the gradient noise halves (variance scales
as $1/B$). To compensate, you can proportionally increase the learning
rate:

$$\eta_{\text{new}} = \eta_{\text{base}} \times \frac{B_{\text{new}}}{B_{\text{base}}}$$

This is the **linear scaling rule** (Goyal et al., 2017). It works well up to
the critical batch size. Beyond it, larger batches degrade generalization
even with tuned learning rates — the model converges to sharper minima.

In [ ]:
# Linear scaling rule example
base_batch_size = 256
base_lr = 3e-4

new_batch_size = 1024
scaled_lr = base_lr * (new_batch_size / base_batch_size)
print(f"Scaled LR: {scaled_lr:.4f}")   # 1.2e-3

### Batch size and generalization

[Smaller batches generally **generalize better**:]{.mark}

1. **Noise as implicit regularization:** mini-batch noise prevents the
   optimizer from converging to sharp, narrow minima (which tend to
   generalize poorly). Large batches converge to sharper minima.
2. **More gradient steps per epoch:** with smaller batches, you see more
   parameter updates per pass over the data, which can help early
   convergence.

For fine-tuning, where datasets are small and you are near a good
initialization (the pretrained weights), small batch sizes (8–32) often
outperform large ones. For pretraining, efficiency dominates and batch
sizes of 256–4096 tokens are typical (actual token count, not sample count).

**NOTE:** See Fig 6 of https://arxiv.org/pdf/1812.06162.

### Token-level batch size for language models

For language models, batch size is most meaningfully measured in **tokens**
(not samples), because samples can have different lengths.

$$\text{tokens per step} = \text{batch\_size} \times \text{seq\_len}$$

Typical pretraining target: 256K–2M tokens per step (achieved via gradient
accumulation when GPU memory limits the physical batch).

In [ ]:
# Token batch size calculation
batch_size   = 8          # samples per GPU
seq_len      = 1024       # tokens per sample
accum_steps  = 4          # gradient accumulation steps
num_gpus     = 1

tokens_per_step = batch_size * seq_len * accum_steps * num_gpus
print(f"Tokens per step: {tokens_per_step:,}")   # 32,768

---

## 6. Choosing a Learning Rate

The learning rate is the single most important hyperparameter. Too large
and the loss diverges; too small and the model trains too slowly or
settles in a poor minimum.

### The learning rate range test

The fastest way to find a reasonable LR: run a sweep from a very small
value (1e-7) to a large one (1e-1) over 100–200 steps, increasing the
LR exponentially. Plot the loss against the LR. The optimal LR is
slightly to the left of where the loss starts to diverge.

In [ ]:
import torch
import torch.nn as nn
import math
import matplotlib.pyplot as plt
from copy import deepcopy

def lr_range_test(
    model,
    dataloader,
    criterion,
    min_lr: float = 1e-7,
    max_lr: float = 0.1,
    num_steps: int = 150,
):
    model_copy = deepcopy(model)
    optimizer  = torch.optim.AdamW(model_copy.parameters(), lr=min_lr,
                                   betas=(0.9, 0.95))
    lrs, losses = [], []

    ratio = (max_lr / min_lr) ** (1 / num_steps)

    data_iter = iter(dataloader)
    for step in range(num_steps):
        try:
            x, y = next(data_iter)
        except StopIteration:
            data_iter = iter(dataloader)
            x, y = next(data_iter)

        optimizer.zero_grad()
        logits = model_copy(x)
        loss   = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
        loss.backward()
        optimizer.step()

        # Record before stepping LR
        current_lr = optimizer.param_groups[0]['lr']
        lrs.append(current_lr)
        losses.append(loss.item())

        # Exponentially increase LR
        for pg in optimizer.param_groups:
            pg['lr'] *= ratio

        if math.isnan(loss.item()) or loss.item() > 10 * losses[0]:
            break

    # Smooth losses with EMA for readability
    smoothed, alpha = [], 0.1
    running = losses[0]
    for l in losses:
        running = alpha * l + (1 - alpha) * running
        smoothed.append(running)

    plt.figure(figsize=(8, 4))
    plt.semilogx(lrs, smoothed)
    plt.xlabel('Learning Rate (log scale)')
    plt.ylabel('Loss (EMA smoothed)')
    plt.title('LR Range Test — pick LR just before the upturn')
    plt.grid(True, which='both', alpha=0.3)
    plt.show()

    return lrs, losses

### Practical LR ranges for LLMs

Based on empirical findings across many LLM training runs:

| Setting | Typical peak LR |
|---|---|
| Pretraining (125M–1B params) | 3e-4 – 6e-4 |
| Pretraining (7B+ params) | 1e-4 – 3e-4 |
| Full fine-tuning (any size) | 1e-5 – 5e-5 |
| LoRA fine-tuning (r=8–64) | 1e-4 – 3e-4 |
| DPO / GRPO | 1e-6 – 5e-6 |

The inverse relationship between model size and learning rate exists because
larger models have more redundancy — a given step size moves a smaller
fraction of the loss landscape. The fine-tuning values are 10–100× smaller
than pretraining because you are starting from a good point and want to
make small, targeted adjustments without destroying pretrained knowledge.

### Why fine-tuning LR matters more than pretraining LR

In pretraining, the model is randomly initialized and far from any
useful minimum. The loss landscape is rough. Large steps are necessary to
make progress, and the model is robust to LR missteps — you might slow
convergence, but you won't break anything fundamentally.

In fine-tuning:
- The pretrained weights encode vast knowledge. An LR that is too large
  performs **catastrophic forgetting**[^catforgetting] — gradient updates overwrite learned
  representations.
- The fine-tuning dataset is small. Large LR causes **overfitting to noise**
  in a few steps.
- With LoRA, the base weights are frozen, but the adapter weights ($A$, $B$)
  start at magnitude ~0. A large LR makes them grow too fast relative to
  the base forward pass.

[^catforgetting]: Catastrophic forgetting occurs when fine-tuning gradient updates are large enough to overwrite the distributed representations learned during pretraining. The pretrained weights encode general language knowledge across all parameters simultaneously — a large update to any one pushes the weight away from the entire knowledge manifold, not just the task-specific part.

In [ ]:
# Catastrophic forgetting demo: compare fine-tuning LR
# Pretrain a small model, then fine-tune with two different LRs

def measure_catastrophic_forgetting(pretrained_model, finetune_data,
                                    eval_data_pretrain, lr):
    """Returns: (finetune_loss_after, pretrain_skill_retention)."""
    model = deepcopy(pretrained_model)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)

    for x, y in finetune_data:          # just a few steps
        opt.zero_grad()
        logits = model(x)
        loss   = nn.functional.cross_entropy(logits.view(-1, logits.size(-1)),
                                             y.view(-1))
        loss.backward()
        opt.step()

    with torch.no_grad():
        retention_losses = []
        for x, y in eval_data_pretrain:
            logits = model(x)
            retention_losses.append(
                nn.functional.cross_entropy(logits.view(-1, logits.size(-1)),
                                            y.view(-1)).item()
            )
    return sum(retention_losses) / len(retention_losses)

---

## 7. Gradient Accumulation

### The problem: effective batch size vs GPU memory

You want a token batch size of 512K tokens (required for stable pretraining
at moderate scale). Your GPU can hold at most 8 sequences of 1024 tokens in
memory — 8,192 tokens. The gap is 64×.

Gradient accumulation solves this by splitting the effective batch into
$k$ **micro-batches**, running a forward/backward pass on each, and only
calling `optimizer.step()` after all $k$ micro-batches. The gradients
accumulate (add) across micro-batches before the update.

$$\text{effective\_batch\_size} = \text{micro\_batch\_size} \times k \times n_{\text{GPU}}$$

Since gradients are additive ($\nabla(\sum_i L_i) = \sum_i \nabla L_i$),
this is mathematically equivalent to a single forward pass over the full
batch — as long as you divide the loss by $k$.

### Why you must divide by $k$

If you do not divide by $k$, the gradient scale grows linearly with the
accumulation steps. After $k$ micro-steps, the accumulated gradient is
$k$ times larger than the single-step gradient, making the effective
learning rate $k\eta$ instead of $\eta$. At large $k$ (e.g., $k=32$),
this causes immediate loss divergence.

$$\text{loss per micro-step} = \frac{L(\mathcal{B}_i)}{k}$$

In [ ]:
# Correct gradient accumulation
def train_step_with_accumulation(model, optimizer, get_batch,
                                 accum_steps: int):
    optimizer.zero_grad()                    # zero ONCE before the loop
    total_loss = 0.0

    for micro_step in range(accum_steps):
        x, y = get_batch()
        logits, loss = model(x, y)

        # Scale the loss by 1/k before backward
        (loss / accum_steps).backward()      # gradients accumulate in .grad

        total_loss += loss.item()

    # After all micro-steps, gradients represent the full effective batch
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    return total_loss / accum_steps

### [The silent `zero_grad` bug]{.underline}

If you call `optimizer.zero_grad()` inside the accumulation loop
(once per micro-step), gradients are wiped between micro-steps. Only the
last micro-step's gradient survives into the `optimizer.step()`:

In [ ]:
# BUG: zero_grad inside the loop
for micro_step in range(accum_steps):
    optimizer.zero_grad()   # ← WRONG: discards all previous accumulation
    x, y = get_batch()
    _, loss = model(x, y)
    (loss / accum_steps).backward()
optimizer.step()
# Result: equivalent to batch_size, not batch_size * accum_steps
# The loss will look fine — this bug is silent

The loss will appear to train normally (because each micro-step still
produces a valid gradient), but the effective batch size is
`batch_size`, not `batch_size * accum_steps`. You will see the model
converge more slowly than expected and may misattribute the problem.

### Verifying accumulation is correct

In [ ]:
# Numerical test: accumulated gradient should equal true batch gradient
torch.manual_seed(0)

batch_size = 32
x_full = torch.randn(batch_size, 128)
y_full = torch.randint(0, 10, (batch_size,))

model = nn.Linear(128, 10)

# --- True large batch ---
model.zero_grad()
loss_true = nn.functional.cross_entropy(model(x_full), y_full)
loss_true.backward()
true_grad = model.weight.grad.clone()

# --- Gradient accumulation (4 micro-batches of 8) ---
model.zero_grad()
for i in range(4):
    x_micro = x_full[i*8:(i+1)*8]
    y_micro = y_full[i*8:(i+1)*8]
    loss_micro = nn.functional.cross_entropy(model(x_micro), y_micro)
    (loss_micro / 4).backward()       # divide by k=4
accum_grad = model.weight.grad.clone()

# Should be identical (to floating-point precision)
max_diff = (true_grad - accum_grad).abs().max().item()
print(f"Max gradient difference: {max_diff:.2e}")   # should be ~1e-7 or less

### Gradient accumulation with mixed precision

When using `torch.autocast`, place the context manager inside the
accumulation loop (around the forward pass), not around the full loop:

In [ ]:
# Correct: autocast per micro-step
optimizer.zero_grad()
for micro_step in range(accum_steps):
    x, y = get_batch()
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        logits, loss = model(x, y)
    (loss / accum_steps).backward()   # backward outside autocast is fine

optimizer.step()

This is correct because `torch.autocast` only affects the forward pass
(and the `loss.item()` call if inside). The backward pass and optimizer
step always happen in FP32.

---

## 8. Optimizer Hyperparameters: Pretraining vs Fine-Tuning

The settings that work for pretraining do not directly transfer to
fine-tuning. Here is a concrete comparison:

| Hyperparameter | Pretraining | Fine-tuning (Full) | Fine-tuning (LoRA) |
|---|---|---|---|
| Optimizer | AdamW | AdamW | AdamW |
| Peak LR | 3e-4 (125M), 1e-4 (1B+) | 1e-5 – 5e-5 | 1e-4 – 3e-4 |
| β₁ | 0.9 | 0.9 | 0.9 |
| β₂ | 0.95 | 0.999 | 0.999 |
| Weight decay | 0.1 | 0.01 – 0.1 | 0.01 |
| Warmup steps | 1–2% of total | 3–10% of total | 10% of total |
| Grad clip | 1.0 | 1.0 | 1.0 |
| Effective batch (tokens) | 256K – 4M | 8K – 128K | 4K – 32K |
| Accum steps | 4–64 | 1–8 | 1–4 |

### Why β₂ changes for fine-tuning

In pretraining, `beta2=0.95` means the second moment forgets the past after
roughly $1/(1-0.95) = 20$ steps. This makes the adaptive scaling responsive
to recent gradient changes across a long run of millions of steps.

In fine-tuning, runs are shorter (hundreds to thousands of steps) and the
gradient magnitudes change less dramatically (you are near a good minimum).
`beta2=0.999` provides smoother, more stable second-moment estimates, which
avoids noisy adaptive scaling that could destabilize the fine-tune.

### Why weight decay is lower for fine-tuning

High weight decay during fine-tuning fights the gradient update:
the model tries to adapt to the new task while decay is constantly
pulling weights toward zero. For full fine-tuning, `weight_decay=0.01`
is common. For LoRA (where you only train the adapter weights, which
start at ~0), even `0.01` can slow convergence — some practitioners
use `0.0` for LoRA adapters.

### Parameter groups in fine-tuning: different LR for different parts

For full fine-tuning, it is common to use a lower LR for early layers
(which encode general features that should change minimally) and a
higher LR for later layers (which encode task-specific representations):

In [ ]:
def make_layerwise_optimizer(model, base_lr, num_layers):
    """
    Linear LR decay: layer 0 gets base_lr * 0.1, last layer gets base_lr.
    Forces early layers to stay close to pretrained representations.
    """
    param_groups = []
    for i, (name, p) in enumerate(model.named_parameters()):
        # Estimate layer depth from parameter name
        layer_idx = 0
        for j in range(num_layers):
            if f'layers.{j}.' in name or f'layer.{j}.' in name:
                layer_idx = j + 1
                break

        lr_scale = 0.1 + 0.9 * (layer_idx / num_layers)   # 0.1 → 1.0
        param_groups.append({
            'params': [p],
            'lr':     base_lr * lr_scale,
            'weight_decay': 0.01 if p.ndim >= 2 else 0.0,
        })

    return torch.optim.AdamW(param_groups, lr=base_lr, betas=(0.9, 0.999))

---

## 9. A Complete Optimizer Setup for Fine-Tuning

Bringing all of the above together into a reusable fine-tuning optimizer
configuration:

In [ ]:
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import LambdaLR
import math
from dataclasses import dataclass, field
from typing import Optional


@dataclass
class OptimizerConfig:
    # Core
    lr:             float = 2e-5
    weight_decay:   float = 0.01
    betas:          tuple = (0.9, 0.999)
    eps:            float = 1e-8
    grad_clip:      float = 1.0

    # Schedule
    warmup_ratio:   float = 0.06      # fraction of steps for warmup
    min_lr_ratio:   float = 0.1       # min_lr = lr * min_lr_ratio

    # Accumulation
    accum_steps:    int   = 1

    # LoRA-specific
    lora_lr_scale:  float = 1.0       # LoRA params can use higher LR if needed


def build_optimizer(model: nn.Module, config: OptimizerConfig,
                    total_steps: int):
    """
    Returns (optimizer, scheduler) ready for a fine-tuning run.
    """
    # Split parameter groups: decay weights, skip biases + norm params
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim >= 2:
            decay.append(p)
        else:
            no_decay.append(p)

    param_groups = [
        {'params': decay,    'weight_decay': config.weight_decay},
        {'params': no_decay, 'weight_decay': 0.0},
    ]

    optimizer = torch.optim.AdamW(
        param_groups,
        lr=config.lr,
        betas=config.betas,
        eps=config.eps,
    )

    # Cosine schedule with warmup
    warmup_steps = int(total_steps * config.warmup_ratio)
    min_ratio    = config.min_lr_ratio

    def lr_lambda(step: int) -> float:
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        progress = min(progress, 1.0)
        return min_ratio + (1.0 - min_ratio) * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = LambdaLR(optimizer, lr_lambda)
    return optimizer, scheduler


def fine_tune_step(model, optimizer, scheduler, get_micro_batch,
                   accum_steps: int, grad_clip: float) -> float:
    """One optimizer step with gradient accumulation."""
    optimizer.zero_grad()
    total_loss = 0.0

    for _ in range(accum_steps):
        x, y = get_micro_batch()
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            logits, loss = model(x, y)
        (loss / accum_steps).backward()
        total_loss += loss.item()

    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()
    scheduler.step()

    return total_loss / accum_steps

---

## Summary

| Concept | Key point |
|---|---|
| SGD | No adaptation; oscillates in ill-conditioned landscapes |
| SGD + momentum | Smooths gradient direction with EMA velocity |
| Adam | Per-parameter adaptive LR via gradient mean and variance |
| AdamW | Decouples weight decay from gradient adaptive scaling |
| L2 vs weight decay | Equivalent for SGD, not for Adam — always use AdamW |
| Batch size | Controls gradient noise; bigger ≠ better past $B^*$ |
| Linear scaling rule | Double batch size → double LR (up to critical batch size) |
| Fine-tuning LR | 10–100× smaller than pretraining; catastrophic forgetting if too large |
| Gradient accumulation | Simulate large batch; divide loss by $k$; zero_grad once before loop |
| β₂ for fine-tuning | Use 0.999 (not 0.95); shorter runs, more stable second moments |

---

## Exercises

**1. The cost of momentum coefficient.** Train a small transformer (Tutorial 2)
with `momentum=0.0, 0.5, 0.9, 0.95, 0.99`. Use the gradient hooks from
Tutorial 4 to plot gradient-to-weight ratio per layer for each run. What
happens to the ratio as momentum increases past 0.9?

**2. Verify the AdamW vs Adam+L2 distinction empirically.** Using the
comparison code in Section 4: plot the per-parameter effective weight decay
(i.e., how much each parameter is actually shrunk per step) for both methods.
Does Adam+L2 apply less decay to parameters with large gradient history?

**3. Find the critical batch size for your nano model.** For batch sizes
64, 128, 256, 512, 1024 (using gradient accumulation to reach the larger
sizes), plot loss vs. training tokens (not steps). At which batch size does
the per-token efficiency peak?

**4. LR sensitivity in fine-tuning vs pretraining.** Take the pretrained
nano model from Tutorial 9. Fine-tune it on a small dataset with LRs
spanning 5e-6 to 5e-4. Measure (a) fine-tune loss after 500 steps and
(b) perplexity on the original pretraining eval set (to detect forgetting).
Plot both as a function of LR. Where is the sweet spot?

**5. Build a gradient accumulation verifier.** Instrument `fine_tune_step`
to compare the accumulated gradient norm to the gradient norm of a true
large batch (when batch fits in memory). Track the ratio across steps — it
should stay close to 1.0. Log it to the dashboard from Tutorial 5.